In [1]:
#Effective energies of the Delta smearing tuning (40 cfgs, 24 operators, 32 sources), pion and proton, one page per pz:
#  meff_<particle>_<gamma>.pdf      diagonal correlator of each operator, gamma = G5, G5G4, G3G5 at sink and source
#  meff_<particle>_GEVP_...pdf      reconstructed 2pt of the 3x3 GEVP over the gammas of each operator
#  ratio_<particle>_GEVP_...pdf     [C_rec(t) / C_rec(t_norm)] of each operator divided by the same ratio of the reference operator
#C = (forward + s_a s_b backward) / 2 with s = +1, -1, +1 for G5, G5G4, G3G5 (tsep = 0: forward only, the proton backward source slice is projected
#with parity_m); the t_src are averaged in each cfg, errors are jackknife over the cfgs. E_eff(t) = log(C(t) / C(t+1)).
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

data_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning"
file_name = "2pt_coulomb_tune_32srcs_40cfgs_24operators_24ops_{particle}_tsrc_fb_tsep20_32src_ncfg40.h5"
plot_dir = f"{data_dir}/plots"
os.makedirs(plot_dir, exist_ok=True)

particles = ["pion", "proton"]
t0 = 2                                  # GEVP: C(t) v = lambda C(t0) v
t_vec = 4                               # eigenvectors at t_vec from the full average, kept fixed for the jackknife samples
t_norm = 3                              # ratio plot: C_rec(t) / C_rec(t_norm)
ref_row = [3.25, 3.25 / 2, 0.7]         # ratio plot: every operator divided by this one
t_max_plot = 16
err_max = 0.1                           # meff points with a larger jackknife error are not drawn (None: all drawn)
ratio_err_max = 0.05                    # the same for the ratio plot; these points do not set its y range either
m_disp = {"pion": 0.141, "proton": None}                   # am of the dispersion line (pion: compare_3rhoz_pion.py); None: no line
ylim_window = {"pion": (0.05, 0.3), "proton": (0.1, 0.5)}   # meff y range: center - lower, center + upper (center: dispersion, or the GEVP plateau)
Ls = 32
gamma_sign = {"G5": 1, "G5G4": -1, "G3G5": 1}             # time reflection C_ab(-t) = s_a s_b C_ab(t), as in the h5 attribute time_reflection


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

def meff(c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.log(c[..., :-1] / c[..., 1:])

def as_str(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load(particle):
    with h5py.File(f"{data_dir}/{file_name.format(particle=particle)}", "r") as f:
        assert as_str(f.attrs["dim_pt2"]) == "cfg, smear, gamma_sink, gamma_source, t_src, pf, tsep", f.attrs["dim_pt2"]
        pt2_forward = f["pt2_forward"][:].mean(axis=4)              # (cfg, smear, g_snk, g_src, pf, tsep), t_src averaged
        pt2_backward = f["pt2_backward"][:].mean(axis=4)
        pf_list = f["pf_list"][:]
        smear_list = f["smear_list"][:]
        gamma_list = [as_str(g) for g in f.attrs["gamma_list"]]
    s = np.array([gamma_sign[g] for g in gamma_list])
    C = 0.5 * (pt2_forward + s[None, None, :, None, None, None] * s[None, None, None, :, None, None] * pt2_backward)
    C[..., 0] = pt2_forward[..., 0]
    return C, pf_list, smear_list, gamma_list


def gevp_rec(C_mean, C_jk):
    #per operator and pz: 3x3 GEVP over the gammas, ground-state eigenvector v0 at (t_vec, t0) normalized to v0^dag C(t0) v0 = 1,
    #reconstructed 2pt v0^dag C(t) v0 of the full average and of every jackknife sample (same v0).
    #C(t0) = U w U^dag; L_inv = w^-1/2 U^dag on the directions with w > 0, so that L_inv C(t0) L_inv^dag = 1: the same as the Cholesky
    #L^-1 when C(t0) is positive definite, and the GEVP in the positive directions when noise makes an eigenvalue of C(t0) negative
    n_cfg, n_smear, _, _, n_pf, n_tsep = C_jk.shape
    rec_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    rec_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    for i in range(n_smear):
        for p in range(n_pf):
            A = C_mean[i, :, :, p]                                          # (g_snk, g_src, tsep)
            A = 0.5 * (A + np.conj(np.swapaxes(A, 0, 1)))                   # hermitian: the G5-G3G5 and G5G4-G3G5 elements are imaginary
            sign = np.sign(np.trace(A[:, :, t0]).real)                      # the pion comes out negative in this contraction convention
            w0, u0 = np.linalg.eigh(sign * A[:, :, t0])
            keep = w0 > 1e-6 * w0.max()
            if not keep.all():
                print(f"  GEVP operator {i}, pz index {p}: C(t0) eigenvalues {np.round(w0 / w0.max(), 6)}, GEVP in the {keep.sum()} positive directions")
            L_inv = (u0[:, keep] / np.sqrt(w0[keep])).conj().T
            w, u = np.linalg.eigh(L_inv @ (sign * A[:, :, t_vec]) @ L_inv.conj().T)
            v0 = L_inv.conj().T @ u[:, np.argmax(w)]
            B = C_jk[:, i, :, :, p]                                         # (cfg, g_snk, g_src, tsep)
            B = 0.5 * (B + np.conj(np.swapaxes(B, 1, 2)))
            rec_mean[i, p] = sign * np.einsum("a,abT,b->T", np.conj(v0), A, v0).real
            rec_jk[:, i, p] = sign * np.einsum("a,cabT,b->cT", np.conj(v0), B, v0).real
    return rec_mean, rec_jk


def styles(smear_list, i_ref):
    #color: rho_T; marker: (rho_T / rho_z, mom_frac); open markers for rho_z = rho_T/3
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink", "tab:olive", "tab:cyan", "black"]
    markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
    rho_T_all = sorted(set(np.round(smear_list[:, 0], 6)))
    shape_all = sorted(set((round(rT / rz, 3), round(fr, 6)) for rT, rz, fr in smear_list))
    out = []
    for i, (rT, rz, fr) in enumerate(smear_list):
        ratio = round(rT / rz, 3)
        rz_name = "rho_T" if ratio == 1 else f"rho_T/{ratio:g}"
        label = f"{i:2d}: rho_T {rT:g}, rho_z = {rz_name}, frac {fr:g}" + ("  (ref)" if i == i_ref else "")
        out.append((colors[rho_T_all.index(round(rT, 6)) % 10], markers[shape_all.index((ratio, round(fr, 6))) % 10], ratio < 3, label))
    return out


def page(pdf, t, y, e, style, title, ylabel, ylim, hline=None, hline_label=None, e_cut=None):
    fig, ax = plt.subplots(figsize=(28, 16), constrained_layout=True)
    n_op = y.shape[0]
    for i in range(n_op):
        color, marker, filled, label = style[i]
        ok = np.isfinite(y[i]) & np.isfinite(e[i]) & (t >= 1) & (t <= t_max_plot)
        if e_cut is not None:
            ok &= e[i] < e_cut
        ax.errorbar(t[ok] + 0.025 * (i - (n_op - 1) / 2), y[i][ok], e[i][ok], fmt=marker + "-", color=color, ms=9, mfc=color if filled else "none",
                    mew=1.5, lw=1.0, elinewidth=1.3, capsize=3, label=label)
    if hline is not None:
        ax.axhline(hline, color="gray", lw=1.5, ls="--", label=hline_label)
    ax.set_xlim(0.5, t_max_plot + 0.5)
    if np.all(np.isfinite(ylim)):
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(1, t_max_plot + 1))
    ax.tick_params(labelsize=18)
    ax.grid(alpha=0.3)
    ax.set_xlabel("t", fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_title(title, fontsize=24)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=15, frameon=False)
    pdf.savefig(fig)
    plt.close(fig)


for particle in particles:
    C, pf_list, smear_list, gamma_list = load(particle)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C.shape
    C_mean = C.mean(axis=0)
    C_jk = (C.sum(axis=0)[None] - C) / (n_cfg - 1)
    i_ref = int(np.where(np.isclose(smear_list, ref_row).all(axis=1))[0][0])
    style = styles(smear_list, i_ref)
    print(f"{particle}: {n_cfg} cfgs, {n_smear} operators, gammas {gamma_list}, pz {pf_list[:, 2].tolist()}, reference operator {i_ref} = {smear_list[i_ref]}")
    t_eff = np.arange(n_tsep - 1)
    t_all = np.arange(n_tsep)
    cut_note = f"   (points with jk error > {err_max} not drawn)" if err_max is not None else ""

    #GEVP over the gammas, reconstructed 2pt
    rec_mean, rec_jk = gevp_rec(C_mean, C_jk)
    meff_rec = meff(rec_mean)                                               # (smear, pf, t)

    #center of the meff y range: dispersion relation, else the GEVP plateau at t = 6..10 (reference operator, all operators, G5-G5)
    def center(p):
        if m_disp[particle] is not None:
            p_lat = 2 * np.pi * pf_list[p][2] / Ls
            return 2 * np.arcsinh(np.sqrt(np.sinh(m_disp[particle] / 2) ** 2 + np.sin(p_lat / 2) ** 2))
        for m in [meff_rec[i_ref, p], meff_rec[:, p], meff(C_mean[:, 0, 0].real)[:, p]]:
            if np.isfinite(m[..., 6:11]).any():
                return np.nanmedian(m[..., 6:11])
        return np.nan

    #meff of each gamma alone, then of the GEVP
    cases = [(g, C_mean[:, ig, ig].real, C_jk[:, :, ig, ig].real) for ig, g in enumerate(gamma_list)]
    for name, c_mean, c_jk in cases + [(f"GEVP_t0{t0}_tvec{t_vec}", rec_mean, rec_jk)]:
        y, e = meff(c_mean), jk(meff(c_jk))[1]
        what = f"{name}-{name}" if name in gamma_list else f"GEVP over {', '.join(gamma_list)} (t0 = {t0}, t_vec = {t_vec}), reconstructed 2pt"
        with PdfPages(f"{plot_dir}/meff_{particle}_{name}.pdf") as pdf:
            for p in range(n_pf):
                c = center(p)
                lo, hi = ylim_window[particle]
                ylim, note = (c - lo, c + hi), ""
                if name == "G3G5" and pf_list[p][2] == 0:                   # no ground-state overlap at rest: y range from the data
                    note = "   [pz = 0: no ground-state overlap" + (", P+ keeps only negative parity]" if particle == "proton" else "]")
                    v = y[:, p, 1:t_max_plot + 1][np.isfinite(y[:, p, 1:t_max_plot + 1]) & (e[:, p, 1:t_max_plot + 1] < (err_max or np.inf))]
                    if v.size:
                        ylim = (np.nanmin([ylim[0], v.min() - 0.05]), np.nanmax([ylim[1], v.max() + 0.05]))
                page(pdf, t_eff, y[:, p], e[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   {n_cfg} cfgs{cut_note}{note}",
                     "E_eff(t) = log C(t)/C(t+1)", ylim, c if m_disp[particle] is not None else None, f"dispersion, am = {m_disp[particle]}", err_max)
        print(f"  wrote {plot_dir}/meff_{particle}_{name}.pdf")

    #ratio plot: [C_rec(t) / C_rec(t_norm)] / [C_rec,ref(t) / C_rec,ref(t_norm)], each jackknife sample divided by its own reference
    with np.errstate(invalid="ignore", divide="ignore"):
        r_mean = rec_mean / rec_mean[:, :, t_norm:t_norm + 1]
        r_mean = r_mean / r_mean[i_ref][None]
        r_jk = rec_jk / rec_jk[:, :, :, t_norm:t_norm + 1]
        r_jk = r_jk / r_jk[:, i_ref][:, None]
    r_err = jk(r_jk)[1]
    with PdfPages(f"{plot_dir}/ratio_{particle}_GEVP_t0{t0}_tvec{t_vec}_tnorm{t_norm}.pdf") as pdf:
        for p in range(n_pf):
            vals, errs = r_mean[:, p, 1:t_max_plot + 1], r_err[:, p, 1:t_max_plot + 1]
            vals = vals[np.isfinite(vals) & (errs < ratio_err_max)]
            lo, hi = np.percentile(vals, [2, 98]) if vals.size else (np.nan, np.nan)
            pad = 0.1 * (hi - lo) + 1e-3
            page(pdf, t_all, r_mean[:, p], r_err[:, p], style,
                 f"{particle}  pz = {pf_list[p][2]}   [C_rec(t)/C_rec({t_norm})] / same for operator {i_ref}   GEVP over gammas, t0 = {t0}, t_vec = {t_vec}"
                 f"   (jk error > {ratio_err_max} not drawn)", "ratio to the reference operator", (lo - pad, hi + pad), 1.0, None, ratio_err_max)
    print(f"  wrote {plot_dir}/ratio_{particle}_GEVP_t0{t0}_tvec{t_vec}_tnorm{t_norm}.pdf")

pion: 40 cfgs, 24 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 9 = [3.25  1.625 0.7  ]
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G5G4.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_G3G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_pion_GEVP_t02_tvec4.pdf
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/ratio_pion_GEVP_t02_tvec4_tnorm3.pdf
proton: 40 cfgs, 24 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 9 = [3.25  1.625 0.7  ]
  wrote /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/plots/meff_proton_G5.pdf
  wrote /Users/mcp3270/Desktop/GLUON

In [2]:
#Smearing tuning run 2 (20 operators, Delta, stream e): effective masses and fit-free overlap comparisons, pion and proton, one page per pz.
#For every particle and channel (G5, G5G4, G3G5 diagonal correlators, and the 3x3 gamma GEVP of each operator), in plot_dir/<particle>/:
#  meff_<channel>.pdf         E_eff(t) = log C(t)/C(t+1) of every operator
#  dmeff_<channel>.pdf        E_eff(t) - E_eff,ref(t) on the same cfgs (correlated jackknife): below 0 = less excited-state contamination than the reference
#  ratio_<channel>.pdf        R(t) = [C(t)/C(t_norm)] / [C_ref(t)/C_ref(t_norm)]: tends to w(t_norm)/w_ref(t_norm), the ground-state fraction at t_norm
#                             relative to the reference (above 1 = more overlap); it rises with t until both operators are ground-state dominated
#  overlap_map_<channel>.pdf  R(T) against the noise (error of E_eff / median over the operators, last usable t), all pz on one page
#and a table on stdout. C = (forward + s_a s_b backward) / 2 with s = +1, -1, +1 for G5, G5G4, G3G5 (tsep = 0: forward only); the t_src are averaged
#in each cfg, errors are jackknife over the cfgs.
import os
import warnings
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

data_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning"
file_name = "2pt_coulomb_tune2_20ops_200cfgs_step6_20ops_{particle}_tsrc_fb_tsep20_32src_ncfg177.h5"
plot_dir = f"{data_dir}/plots_run2"

particles = ["pion", "proton"]
t0 = 3                                  # gamma GEVP: C(t) v = lambda C(t0) v
t_vec = 6                              # eigenvectors at t_vec from the full average, kept fixed for the jackknife samples
t_norm = 3                              # ratio: C(t) / C(t_norm)
ref_row = [5.0, 5.0 / 3, 0.8]           # reference operator (run-1 op 23, in both runs)
t_max_plot = 16
err_max = 0.1                           # meff points with a larger jackknife error are not drawn
diff_err_max = 0.05                     # the same for the E_eff difference and the ratio
usable_err = 0.05                       # a t is usable when the median E_eff error over the operators is below this
m_disp = {"pion": 0.141, "proton": None}                   # am of the dispersion line (pion: compare_3rhoz_pion.py); None: no line
ylim_window = {"pion": (0.05, 0.3), "proton": (0.1, 0.5)}   # meff y range: center - lower, center + upper (center: dispersion, or the GEVP plateau)
Ls = 32
gamma_sign = {"G5": 1, "G5G4": -1, "G3G5": 1}             # time reflection C_ab(-t) = s_a s_b C_ab(t), as in the h5 attribute time_reflection
noise_cut = 1.2                         # table: operators ranked by overlap among those with noise <= noise_cut (the run-1 cut)
map_noise_max = 3.0                     # overlap map: operators noisier than this (no signal at that pz) are listed, not drawn
warnings.filterwarnings("ignore", message="All-NaN slice encountered")     # t with no signal in any operator
warnings.filterwarnings("ignore", message="Mean of empty slice")


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

def meff(c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.log(c[..., :-1] / c[..., 1:])

def as_str(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load(particle):
    with h5py.File(f"{data_dir}/{file_name.format(particle=particle)}", "r") as f:
        assert as_str(f.attrs["dim_pt2"]) == "cfg, smear, gamma_sink, gamma_source, t_src, pf, tsep", f.attrs["dim_pt2"]
        pt2_forward = f["pt2_forward"][:].mean(axis=4)              # (cfg, smear, g_snk, g_src, pf, tsep), t_src averaged
        pt2_backward = f["pt2_backward"][:].mean(axis=4)
        pf_list = f["pf_list"][:]
        smear_list = f["smear_list"][:]
        gamma_list = [as_str(g) for g in f.attrs["gamma_list"]]
    s = np.array([gamma_sign[g] for g in gamma_list])
    C = 0.5 * (pt2_forward + s[None, None, :, None, None, None] * s[None, None, None, :, None, None] * pt2_backward)
    C[..., 0] = pt2_forward[..., 0]
    return C, pf_list, smear_list, gamma_list


def gevp_rec(C_mean, C_jk):
    #per operator and pz: 3x3 GEVP over the gammas, ground-state eigenvector v0 at (t_vec, t0) normalized to v0^dag C(t0) v0 = 1,
    #reconstructed 2pt v0^dag C(t) v0 of the full average and of every jackknife sample (same v0).
    #C(t0) = U w U^dag; L_inv = w^-1/2 U^dag on the directions with w > 0 (the Cholesky L^-1 when C(t0) is positive definite)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C_jk.shape
    rec_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    rec_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    for i in range(n_smear):
        for p in range(n_pf):
            A = C_mean[i, :, :, p]                                          # (g_snk, g_src, tsep)
            A = 0.5 * (A + np.conj(np.swapaxes(A, 0, 1)))                   # hermitian: the G5-G3G5 and G5G4-G3G5 elements are imaginary
            sign = np.sign(np.trace(A[:, :, t0]).real)                      # the pion comes out negative in this contraction convention
            w0, u0 = np.linalg.eigh(sign * A[:, :, t0])
            keep = w0 > 1e-6 * w0.max()
            if not keep.all():
                print(f"  GEVP operator {i}, pz index {p}: C(t0) eigenvalues {np.round(w0 / w0.max(), 6)}, GEVP in the {keep.sum()} positive directions")
            L_inv = (u0[:, keep] / np.sqrt(w0[keep])).conj().T
            w, u = np.linalg.eigh(L_inv @ (sign * A[:, :, t_vec]) @ L_inv.conj().T)
            v0 = L_inv.conj().T @ u[:, np.argmax(w)]
            B = C_jk[:, i, :, :, p]                                         # (cfg, g_snk, g_src, tsep)
            B = 0.5 * (B + np.conj(np.swapaxes(B, 1, 2)))
            rec_mean[i, p] = sign * np.einsum("a,abT,b->T", np.conj(v0), A, v0).real
            rec_jk[:, i, p] = sign * np.einsum("a,cabT,b->cT", np.conj(v0), B, v0).real
    return rec_mean, rec_jk


def styles(smear_list, i_ref):
    #one colour map and marker per group of consecutive rows of the same type (rho_z / rho_T, mom_frac), darker for larger rho_T; the reference in black
    cmaps = ["Blues", "Oranges", "Greens", "Purples", "Reds", "YlOrBr"]
    markers = ["o", "s", "^", "D", "v", "P"]
    kind = [(round(rz / rT, 3), round(fr, 6)) for rT, rz, fr in smear_list]
    group, k, prev = [], -1, None
    for i in range(len(smear_list)):
        if i != i_ref:
            k += prev is None or kind[i] != kind[prev]
            prev = i
        group.append(k)
    out = []
    for i, (rT, rz, fr) in enumerate(smear_list):
        ratio = round(rT / rz, 3)
        rz_name = "rho_T" if ratio == 1 else f"rho_T/{ratio:g}"
        label = f"{i:2d}: [{rT:g}, {rz_name}, {fr:g}]"
        if i == i_ref:
            out.append(("black", "*", label + "  (ref)"))
            continue
        members = sorted((j for j in range(len(smear_list)) if group[j] == group[i] and j != i_ref), key=lambda j: smear_list[j][0])
        shade = 0.45 + 0.5 * (members.index(i) / max(len(members) - 1, 1))
        out.append((plt.get_cmap(cmaps[group[i] % len(cmaps)])(shade), markers[group[i] % len(markers)], label))
    return out


def page(pdf, t, y, e, style, title, ylabel, ylim, hline=None, hline_label=None, e_cut=None):
    fig, ax = plt.subplots(figsize=(28, 16), constrained_layout=True)
    n_op = y.shape[0]
    for i in range(n_op):
        color, marker, label = style[i]
        ok = np.isfinite(y[i]) & np.isfinite(e[i]) & (t >= 1) & (t <= t_max_plot)
        if e_cut is not None:
            ok &= e[i] < e_cut
        ax.errorbar(t[ok] + 0.02 * (i - (n_op - 1) / 2), y[i][ok], e[i][ok], fmt=marker + "-", color=color, ms=12 if marker == "*" else 9,
                    mec="black" if marker == "*" else color, mew=1.2, lw=2.0 if marker == "*" else 1.0, elinewidth=1.3, capsize=3, label=label)
    if hline is not None:
        ax.axhline(hline, color="gray", lw=1.5, ls="--", label=hline_label)
    ax.set_xlim(0.5, t_max_plot + 0.5)
    if np.all(np.isfinite(ylim)):
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(1, t_max_plot + 1))
    ax.tick_params(labelsize=18)
    ax.grid(alpha=0.3)
    ax.set_xlabel("t", fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_title(title, fontsize=24)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=15, frameon=False)
    pdf.savefig(fig)
    plt.close(fig)


def range_of(vals, errs, e_cut, pad_frac=0.1):
    v = vals[np.isfinite(vals) & np.isfinite(errs) & (errs < e_cut)]
    if not v.size:
        return (np.nan, np.nan)
    lo, hi = np.percentile(v, [2, 98])
    pad = pad_frac * (hi - lo) + 1e-3
    return (lo - pad, hi + pad)


for particle in particles:
    C, pf_list, smear_list, gamma_list = load(particle)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C.shape
    C_mean = C.mean(axis=0)
    C_jk = (C.sum(axis=0)[None] - C) / (n_cfg - 1)
    match = np.where(np.isclose(smear_list, ref_row).all(axis=1))[0]
    assert len(match) == 1, f"ref_row {ref_row} is not one of the operators in the file"
    i_ref = int(match[0])
    style = styles(smear_list, i_ref)
    out_dir = f"{plot_dir}/{particle}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{particle}: {n_cfg} cfgs, {n_smear} operators, gammas {gamma_list}, pz {pf_list[:, 2].tolist()}, reference operator {i_ref} = {smear_list[i_ref]}")
    t_eff = np.arange(n_tsep - 1)
    t_all = np.arange(n_tsep)

    rec_mean, rec_jk = gevp_rec(C_mean, C_jk)
    channels = {g: (C_mean[:, ig, ig].real, C_jk[:, :, ig, ig].real) for ig, g in enumerate(gamma_list)}
    channels["GEVP"] = (rec_mean, rec_jk)
    meff_rec = meff(rec_mean)

    #center of the meff y range: dispersion relation, else the GEVP plateau at t = 6..10 (reference operator, all operators, G5-G5)
    def center(p):
        if m_disp[particle] is not None:
            p_lat = 2 * np.pi * pf_list[p][2] / Ls
            return 2 * np.arcsinh(np.sqrt(np.sinh(m_disp[particle] / 2) ** 2 + np.sin(p_lat / 2) ** 2))
        for m in [meff_rec[i_ref, p], meff_rec[:, p], meff(C_mean[:, 0, 0].real)[:, p]]:
            if np.isfinite(m[..., 6:11]).any():
                return np.nanmedian(m[..., 6:11])
        return np.nan

    for name, (c_mean, c_jk) in channels.items():
        what = f"{name}-{name}" if name in gamma_list else f"GEVP over {', '.join(gamma_list)} (t0 = {t0}, t_vec = {t_vec}), reconstructed 2pt"
        y, yj = meff(c_mean), meff(c_jk)
        e = jk(yj)[1]
        dy = y - y[i_ref][None]
        de = jk(yj - yj[:, i_ref][:, None])[1]
        with np.errstate(invalid="ignore", divide="ignore"):
            r = (c_mean / c_mean[:, :, t_norm:t_norm + 1]) / (c_mean[i_ref] / c_mean[i_ref, :, t_norm:t_norm + 1])[None]
            rj = (c_jk / c_jk[:, :, :, t_norm:t_norm + 1]) / (c_jk[:, i_ref] / c_jk[:, i_ref, :, t_norm:t_norm + 1])[:, None]
        re = jk(rj)[1]

        #per pz: usable t (median E_eff error below usable_err), noise = geometric mean of err/median over the last 4 usable t <= 10, overlap R at T
        noise = np.full((n_smear, n_pf), np.nan)
        R_T, R_T_err, T_used = np.full((n_smear, n_pf), np.nan), np.full((n_smear, n_pf), np.nan), {}
        for p in range(n_pf):
            med = np.nanmedian(e[:, p], axis=0)
            usable = [t for t in range(t_norm, 11) if np.isfinite(med[t]) and med[t] <= usable_err]
            if not usable:
                continue
            win = usable[-4:]
            with np.errstate(invalid="ignore", divide="ignore"):
                noise[:, p] = np.exp(np.nanmean(np.log(e[:, p, win] / med[win][None]), axis=1))
            T = min(7, usable[-1] + 1)
            if T >= t_norm + 2:                                             # R at t_norm + 1 is one step of E_eff: no overlap information
                T_used[p] = T
                R_T[:, p], R_T_err[:, p] = r[:, p, T], re[:, p, T]

        note0 = {p: ("   [pz = 0: no ground-state overlap" + (", P+ keeps only negative parity]" if particle == "proton" else "]"))
                 if name == "G3G5" and pf_list[p][2] == 0 else "" for p in range(n_pf)}
        with PdfPages(f"{out_dir}/meff_{name}.pdf") as pdf:
            for p in range(n_pf):
                c = center(p)
                lo, hi = ylim_window[particle]
                ylim = (c - lo, c + hi) if not note0[p] else range_of(y[:, p, 1:t_max_plot + 1], e[:, p, 1:t_max_plot + 1], err_max)
                page(pdf, t_eff, y[:, p], e[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff   {n_cfg} cfgs{note0[p]}",
                     "E_eff(t) = log C(t)/C(t+1)", ylim, c if m_disp[particle] is not None else None, f"dispersion, am = {m_disp[particle]}", err_max)
        with PdfPages(f"{out_dir}/dmeff_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_eff, dy[:, p], de[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff - E_eff(op {i_ref}), same cfgs{note0[p]}",
                     "E_eff(t) - E_eff,ref(t)    (< 0: less excited-state contamination)", range_of(dy[:, p, 1:t_max_plot + 1], de[:, p, 1:t_max_plot + 1], diff_err_max),
                     0.0, None, diff_err_max)
        with PdfPages(f"{out_dir}/ratio_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_all, r[:, p], re[:, p], style,
                     f"{particle}  pz = {pf_list[p][2]}   {what}   [C(t)/C({t_norm})] / same for op {i_ref}{note0[p]}",
                     f"ratio -> ground-state fraction at t = {t_norm} relative to op {i_ref}   (> 1: more overlap)",
                     range_of(r[:, p, 1:t_max_plot + 1], re[:, p, 1:t_max_plot + 1], diff_err_max), 1.0, None, diff_err_max)

        #overlap map: all pz on one page
        fig, axes = plt.subplots(2, 4, figsize=(30, 15), constrained_layout=True)
        for p in range(n_pf):
            ax = axes.flat[p]
            ax.grid(alpha=0.3)
            ax.axhline(1, color="gray", lw=1)
            ax.axvline(1, color="gray", lw=1, ls=":")
            ax.axvline(noise_cut, color="gray", lw=1, ls="--")
            ok = np.isfinite(R_T[:, p]) & np.isfinite(noise[:, p])
            for i in np.where(ok & (noise[:, p] <= map_noise_max))[0]:
                color, marker, label = style[i]
                x = max(noise[i, p], 0.5)
                ax.errorbar(x, R_T[i, p], R_T_err[i, p], fmt=marker, color=color, ms=14 if marker == "*" else 10, mec="black", mew=0.8, elinewidth=1.2, capsize=0)
                ax.annotate(str(i), (x, R_T[i, p]), xytext=(5, 4), textcoords="offset points", fontsize=11)
            hidden = np.where(ok & (noise[:, p] > map_noise_max))[0]
            if hidden.size:
                ax.text(0.98, 0.02, f"noise > {map_noise_max:g}, not drawn: " + ", ".join(f"{i} ({noise[i, p]:.0f})" for i in hidden), transform=ax.transAxes,
                        ha="right", va="bottom", fontsize=12, bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
            ax.set_xscale("log")
            ax.set_xlim(0.5, map_noise_max)
            ax.set_xticks([0.5, 0.7, 1, noise_cut, 1.5, 2, 3])
            ax.set_xticklabels(["0.5", "0.7", "1", f"{noise_cut:g}", "1.5", "2", "3"])
            ax.minorticks_off()
            ax.tick_params(labelsize=14)
            ax.set_title(f"pz = {pf_list[p][2]}   (R at T = {T_used.get(p, '-')})" + (note0[p] and "  no ground-state overlap"), fontsize=17)
            if p % 4 == 0:
                ax.set_ylabel(f"overlap: ground-state fraction at t = {t_norm} relative to op {i_ref}", fontsize=15)
            if p >= 3:
                ax.set_xlabel("noise: E_eff error / median over the operators", fontsize=15)
        leg = axes.flat[7]
        leg.axis("off")
        handles = [plt.Line2D([], [], marker=m, ls="", color=c, mec="black", mew=0.8, ms=14 if m == "*" else 10, label=l) for c, m, l in style]
        leg.legend(handles=handles, loc="center", fontsize=13, frameon=False)
        fig.suptitle(f"{particle}   {what}   {n_cfg} cfgs: top-left is best (more overlap, less noise)", fontsize=22)
        with PdfPages(f"{out_dir}/overlap_map_{name}.pdf") as pdf:
            pdf.savefig(fig)
        plt.close(fig)

        print(f"  {name}: op:R/noise, R = overlap relative to op {i_ref} at T, sorted by R among the operators with noise <= {noise_cut:g}")
        for p in range(n_pf):
            if p not in T_used:
                print(f"    pz {pf_list[p][2]}: no usable t")
                continue
            good = [i for i in np.argsort(-R_T[:, p]) if np.isfinite(R_T[i, p]) and noise[i, p] <= noise_cut]
            bad = [i for i in range(n_smear) if i not in good]
            print(f"    pz {pf_list[p][2]} (T = {T_used[p]}): " + "  ".join(f"{i}:{R_T[i, p]:.3f}/{noise[i, p]:.2f}" for i in good[:7])
                  + f"   | noise > {noise_cut:g}: " + " ".join(map(str, bad)))
    print(f"  wrote {out_dir}/")


pion: 177 cfgs, 20 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 19 = [5.         1.66666667 0.8       ]
  GEVP operator 15, pz index 6: C(t0) eigenvalues [-0.004867  0.216116  1.      ], GEVP in the 2 positive directions
  G5: op:R/noise, R = overlap relative to op 19 at T, sorted by R among the operators with noise <= 1.2
    pz 0 (T = 7): 13:1.014/1.09  17:1.010/1.15  12:1.004/1.00  16:1.004/1.06  11:1.004/1.11  19:1.000/1.10  10:1.000/1.05   | noise > 1.2: 14 15 18
    pz 1 (T = 7): 18:1.031/1.17  13:1.026/1.19  17:1.020/1.09  12:1.008/1.09  11:1.008/1.07  16:1.008/1.02  10:1.000/1.02   | noise > 1.2: 14 15
    pz 2 (T = 7): 18:1.057/0.98  17:1.039/1.00  11:1.018/0.71  16:1.018/1.00  10:1.006/0.74  19:1.000/0.98  9:0.992/0.76   | noise > 1.2: 1 2 3 4 12 13 14 15
    pz 3 (T = 7): 18:1.074/1.00  17:1.053/1.00  11:1.035/0.68  16:1.028/0.99  10:1.020/0.69  9:1.003/0.71  19:1.000/0.90   | noise > 1.2: 0 1 2 3 4 12 13 14 15
    pz 4 (T = 6): 18:

In [3]:
#Smearing tuning run 2 (20 operators, Delta, stream e): effective masses and fit-free overlap comparisons, pion and proton, one page per pz.
#For every particle and channel (G5, G5G4, G3G5 diagonal correlators, and the 3x3 gamma GEVP of each operator with solve_gevp of
#compare_3rhoz_pion.py: GEVP = reconstructed 2pt v0^dag C(t) v0, GEVP_lambda = principal correlator lambda_0(t, t0)), in plot_dir/<particle>/:
#  meff_<channel>.pdf         E_eff(t) = log C(t)/C(t+1) of every operator
#  dmeff_<channel>.pdf        E_eff(t) - E_eff,ref(t) on the same cfgs (correlated jackknife): below 0 = less excited-state contamination than the reference
#The overlap pages below are not made for GEVP_lambda: lambda_0 is normalized by C(t0) and carries no overlap factor.
#  ratio_<channel>.pdf        R(t) = [C(t)/C(t_norm)] / [C_ref(t)/C_ref(t_norm)]: tends to w(t_norm)/w_ref(t_norm), the ground-state fraction at t_norm
#                             relative to the reference (above 1 = more overlap); R(t_norm) = 1, it rises (falls) while E_eff(t) is below (above)
#                             E_eff,ref(t), and is flat once both operators are ground-state dominated
#  overlap_map_<channel>.pdf  R(T) against the noise (geometric mean over the last 4 usable t <= 10 of the E_eff error / median over the operators),
#                             all pz on one page
#and a table on stdout. C = (forward + s_a s_b backward) / 2 with s = +1, -1, +1 for G5, G5G4, G3G5 (tsep = 0: forward only), made hermitian in the
#gammas and multiplied by c_sign so that it is positive at tsep >= 1 (the G3G5 diagonal at tsep = 0 is a contact term of the other sign, t = 0 is
#not used); the t_src are averaged in each cfg, errors are jackknife over the cfgs.
import os
import warnings
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

data_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning"
file_name = "2pt_coulomb_tune2_20ops_200cfgs_step6_20ops_{particle}_tsrc_fb_tsep20_32src_ncfg177.h5"
plot_dir = f"{data_dir}/plots_run2"

particles = ["pion", "proton"]
t0 = 2                                  # gamma GEVP: C(t) v = lambda C(t0) v
t_vec = 4                               # reconstructed 2pt: eigenvector at t_vec from the full average, kept fixed for the jackknife samples
t_norm = 3                              # ratio: C(t) / C(t_norm)
ref_row = [5.0, 5.0 / 3, 0.8]           # reference operator (run-1 op 23, in both runs)
t_max_plot = 16
err_max = 0.1                           # meff points with a larger jackknife error are not drawn
diff_err_max = 0.05                     # the same for the E_eff difference and the ratio
usable_err = 0.05                       # a t is usable when the median E_eff error over the operators is below this
m_disp = {"pion": 0.141, "proton": None}                   # am of the dispersion line (pion: compare_3rhoz_pion.py); None: no line
ylim_window = {"pion": (0.05, 0.3), "proton": (0.1, 0.5)}   # meff y range: center - lower, center + upper (center: dispersion, or the GEVP plateau)
Ls = 32
gamma_sign = {"G5": 1, "G5G4": -1, "G3G5": 1}             # time reflection C_ab(-t) = s_a s_b C_ab(t), as in the h5 attribute time_reflection
c_sign = {"pion": -1, "proton": 1}                        # the pion 2pt comes out negative in this contraction convention (all gammas, operators, pz, tsep >= 1)
noise_cut = 1.2                         # table: operators ranked by overlap among those with noise <= noise_cut (the run-1 cut)
map_noise_max = 3.0                     # overlap map: operators noisier than this (no signal at that pz) are listed, not drawn
warnings.filterwarnings("ignore", message="All-NaN slice encountered")     # t with no signal in any operator
warnings.filterwarnings("ignore", message="Mean of empty slice")


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

def meff(c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.log(c[..., :-1] / c[..., 1:])

def as_str(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load(particle):
    #C (cfg, smear, g_snk, g_src, pf, tsep): forward-backward average, hermitian in (g_snk, g_src), times c_sign
    with h5py.File(f"{data_dir}/{file_name.format(particle=particle)}", "r") as f:
        assert as_str(f.attrs["dim_pt2"]) == "cfg, smear, gamma_sink, gamma_source, t_src, pf, tsep", f.attrs["dim_pt2"]
        pt2_forward = f["pt2_forward"][:].mean(axis=4)              # (cfg, smear, g_snk, g_src, pf, tsep), t_src averaged
        pt2_backward = f["pt2_backward"][:].mean(axis=4)
        pf_list = f["pf_list"][:]
        smear_list = f["smear_list"][:]
        gamma_list = [as_str(g) for g in f.attrs["gamma_list"]]
    s = np.array([gamma_sign[g] for g in gamma_list])
    C = 0.5 * (pt2_forward + s[None, None, :, None, None, None] * s[None, None, None, :, None, None] * pt2_backward)
    C[..., 0] = pt2_forward[..., 0]
    C = 0.5 * (C + np.conj(np.swapaxes(C, 2, 3)))                   # the G5-G3G5 and G5G4-G3G5 elements are imaginary
    C = c_sign[particle] * C
    return C, pf_list, smear_list, gamma_list


def solve_gevp(C_matrix):
    #from compare_3rhoz_pion.py. C_matrix (op_snk, op_src, pf, tsep), hermitian and positive; here op = gamma (n_op = 3) for one smearing.
    #GEVP at every t in t_list by Cholesky of C(t0): eigenvalues in decreasing order, eigenvectors v normalized to v^dag C(t0) v = 1.
    #Added: a pz whose C(t0) is not positive definite (no signal, e.g. the wide unboosted operators at large pz) is left NaN instead of stopping the run.
    eigen_val = np.full((n_pf, n_op, len(t_list)), np.nan)
    eigen_vec = np.full((n_pf, n_op, len(t_list), n_op), np.nan, "<c16")
    for ipf in range(n_pf):
        try:
            L = np.linalg.cholesky(C_matrix[:, :, ipf, t0])
        except np.linalg.LinAlgError:
            continue
        L_inv = np.linalg.inv(L)
        for it, t in enumerate(t_list):
            M = L_inv @ C_matrix[:, :, ipf, t] @ L_inv.conj().T
            w, u = np.linalg.eigh(M)
            v = L_inv.conj().T @ u
            order = np.argsort(w)[::-1]
            eigen_val[ipf, :, it] = w[order]
            eigen_vec[ipf, :, it] = v[:, order].T
    return eigen_val, eigen_vec


def gevp(C_mean, C_jk):
    #per smearing: reconstructed 2pt v0^dag C(t) v0 with the ground-state eigenvector v0 of the full average at t_vec (the same v0 for every jackknife
    #sample), and the principal correlator lambda_0(t), solved again on every jackknife sample; lambda_0 is NaN at t <= t0 (t_list starts at t0 + 1)
    rec_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    rec_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    lam_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    lam_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    it_vec = list(t_list).index(t_vec)
    for i in range(n_smear):
        eigen_val, eigen_vec = solve_gevp(C_mean[i])
        v0 = eigen_vec[:, 0, it_vec, :]                                     # (pf, gamma)
        rec_mean[i] = np.einsum("pa,abpT,pb->pT", np.conj(v0), C_mean[i], v0).real
        rec_jk[:, i] = np.einsum("pa,cabpT,pb->cpT", np.conj(v0), C_jk[:, i], v0).real
        lam_mean[i, :, t0 + 1:] = eigen_val[:, 0]
        lam_jk[:, i, :, t0 + 1:] = np.array([solve_gevp(C_jk[icfg, i])[0][:, 0] for icfg in range(n_cfg)])
        for ipf in np.where(~np.isfinite(eigen_val[:, 0, 0]))[0]:
            w0 = np.linalg.eigvalsh(C_mean[i, :, :, ipf, t0])
            print(f"  GEVP operator {i}, pz {pf_list[ipf][2]}: C(t0) not positive definite (eigenvalues / largest {np.round(w0 / w0.max(), 6)}), left NaN")
        n_fail = np.sum(~np.isfinite(lam_jk[:, i, :, t0 + 1]) & np.isfinite(eigen_val[None, :, 0, 0]))
        if n_fail:
            print(f"  GEVP operator {i}: C(t0) not positive definite on {n_fail} jackknife samples (sample, pz), their lambda_0 is NaN")
    deviation = np.nanmax(np.abs(rec_mean[:, :, t0] - 1))
    assert deviation < 1e-10, f"v0 is not normalized to v0^dag C(t0) v0 = 1 (max deviation {deviation:.1e})"
    return rec_mean, rec_jk, lam_mean, lam_jk


def styles(smear_list, i_ref):
    #one colour map and marker per group of consecutive rows of the same type (rho_z / rho_T, mom_frac), darker for larger rho_T; the reference in black
    cmaps = ["Blues", "Oranges", "Greens", "Purples", "Reds", "YlOrBr"]
    markers = ["o", "s", "^", "D", "v", "P"]
    kind = [(round(rz / rT, 3), round(fr, 6)) for rT, rz, fr in smear_list]
    group, k, prev = [], -1, None
    for i in range(len(smear_list)):
        if i != i_ref:
            k += prev is None or kind[i] != kind[prev]
            prev = i
        group.append(k)
    out = []
    for i, (rT, rz, fr) in enumerate(smear_list):
        ratio = round(rT / rz, 3)
        rz_name = "rho_T" if ratio == 1 else f"rho_T/{ratio:g}"
        label = f"{i:2d}: [{rT:g}, {rz_name}, {fr:g}]"
        if i == i_ref:
            out.append(("black", "*", label + "  (ref)"))
            continue
        members = sorted((j for j in range(len(smear_list)) if group[j] == group[i] and j != i_ref), key=lambda j: smear_list[j][0])
        shade = 0.45 + 0.5 * (members.index(i) / max(len(members) - 1, 1))
        out.append((plt.get_cmap(cmaps[group[i] % len(cmaps)])(shade), markers[group[i] % len(markers)], label))
    return out


def page(pdf, t, y, e, style, title, ylabel, ylim, hline=None, hline_label=None, e_cut=None):
    fig, ax = plt.subplots(figsize=(28, 16), constrained_layout=True)
    n_op = y.shape[0]
    for i in range(n_op):
        color, marker, label = style[i]
        ok = np.isfinite(y[i]) & np.isfinite(e[i]) & (t >= 1) & (t <= t_max_plot)
        if e_cut is not None:
            ok &= e[i] < e_cut
        ax.errorbar(t[ok] + 0.02 * (i - (n_op - 1) / 2), y[i][ok], e[i][ok], fmt=marker + "-", color=color, ms=12 if marker == "*" else 9,
                    mec="black" if marker == "*" else color, mew=1.2, lw=2.0 if marker == "*" else 1.0, elinewidth=1.3, capsize=3, label=label)
    if hline is not None:
        ax.axhline(hline, color="gray", lw=1.5, ls="--", label=hline_label)
    ax.set_xlim(0.5, t_max_plot + 0.5)
    if np.all(np.isfinite(ylim)):
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(1, t_max_plot + 1))
    ax.tick_params(labelsize=18)
    ax.grid(alpha=0.3)
    ax.set_xlabel("t", fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_title(title, fontsize=24)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=15, frameon=False)
    pdf.savefig(fig)
    plt.close(fig)


def range_of(vals, errs, e_cut, pad_frac=0.1):
    v = vals[np.isfinite(vals) & np.isfinite(errs) & (errs < e_cut)]
    if not v.size:
        return (np.nan, np.nan)
    lo, hi = np.percentile(v, [2, 98])
    pad = pad_frac * (hi - lo) + 1e-3
    return (lo - pad, hi + pad)


for particle in particles:
    C, pf_list, smear_list, gamma_list = load(particle)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C.shape
    C_mean = C.mean(axis=0)
    C_jk = (C.sum(axis=0)[None] - C) / (n_cfg - 1)
    match = np.where(np.isclose(smear_list, ref_row).all(axis=1))[0]
    assert len(match) == 1, f"ref_row {ref_row} is not one of the operators in the file"
    i_ref = int(match[0])
    style = styles(smear_list, i_ref)
    out_dir = f"{plot_dir}/{particle}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{particle}: {n_cfg} cfgs, {n_smear} operators, gammas {gamma_list}, pz {pf_list[:, 2].tolist()}, reference operator {i_ref} = {smear_list[i_ref]}")
    t_eff = np.arange(n_tsep - 1)
    t_all = np.arange(n_tsep)

    n_op = len(gamma_list)                                                  # GEVP basis of solve_gevp: the gammas of one smearing
    t_list = np.arange(t0 + 1, n_tsep)
    rec_mean, rec_jk, lam_mean, lam_jk = gevp(C_mean, C_jk)
    channels = {g: (C_mean[:, ig, ig].real, C_jk[:, :, ig, ig].real) for ig, g in enumerate(gamma_list)}
    channels["GEVP"] = (rec_mean, rec_jk)
    channels["GEVP_lambda"] = (lam_mean, lam_jk)
    meff_rec = meff(rec_mean)

    #center of the meff y range: dispersion relation, else the GEVP plateau at t = 6..10 (reference operator, all operators, G5-G5)
    def center(p):
        if m_disp[particle] is not None:
            p_lat = 2 * np.pi * pf_list[p][2] / Ls
            return 2 * np.arcsinh(np.sqrt(np.sinh(m_disp[particle] / 2) ** 2 + np.sin(p_lat / 2) ** 2))
        for m in [meff_rec[i_ref, p], meff_rec[:, p], meff(C_mean[:, 0, 0].real)[:, p]]:
            if np.isfinite(m[..., 6:11]).any():
                return np.nanmedian(m[..., 6:11])
        return np.nan

    for name, (c_mean, c_jk) in channels.items():
        what = {"GEVP": f"GEVP over {', '.join(gamma_list)} (t0 = {t0}, t_vec = {t_vec}), reconstructed 2pt",
                "GEVP_lambda": f"GEVP over {', '.join(gamma_list)} (t0 = {t0}), principal correlator lambda_0"}.get(name, f"{name}-{name}")
        y, yj = meff(c_mean), meff(c_jk)
        e = jk(yj)[1]
        dy = y - y[i_ref][None]
        de = jk(yj - yj[:, i_ref][:, None])[1]
        note0 = {p: ("   [pz = 0: no ground-state overlap" + (", P+ keeps only negative parity]" if particle == "proton" else "]"))
                 if name == "G3G5" and pf_list[p][2] == 0 else "" for p in range(n_pf)}
        with PdfPages(f"{out_dir}/meff_{name}.pdf") as pdf:
            for p in range(n_pf):
                c = center(p)
                lo, hi = ylim_window[particle]
                ylim = (c - lo, c + hi) if not note0[p] else range_of(y[:, p, 1:t_max_plot + 1], e[:, p, 1:t_max_plot + 1], err_max)
                page(pdf, t_eff, y[:, p], e[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff   {n_cfg} cfgs{note0[p]}",
                     "E_eff(t) = log lambda_0(t)/lambda_0(t+1)" if name == "GEVP_lambda" else "E_eff(t) = log C(t)/C(t+1)", ylim,
                     c if m_disp[particle] is not None else None, f"dispersion, am = {m_disp[particle]}", err_max)
        with PdfPages(f"{out_dir}/dmeff_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_eff, dy[:, p], de[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff - E_eff(op {i_ref}), same cfgs{note0[p]}",
                     "E_eff(t) - E_eff,ref(t)    (< 0: less excited-state contamination)", range_of(dy[:, p, 1:t_max_plot + 1], de[:, p, 1:t_max_plot + 1], diff_err_max),
                     0.0, None, diff_err_max)
        if name == "GEVP_lambda":
            continue

        with np.errstate(invalid="ignore", divide="ignore"):
            r = (c_mean / c_mean[:, :, t_norm:t_norm + 1]) / (c_mean[i_ref] / c_mean[i_ref, :, t_norm:t_norm + 1])[None]
            rj = (c_jk / c_jk[:, :, :, t_norm:t_norm + 1]) / (c_jk[:, i_ref] / c_jk[:, i_ref, :, t_norm:t_norm + 1])[:, None]
        re = jk(rj)[1]

        #per pz: usable t (median E_eff error below usable_err), noise = geometric mean of err/median over the last 4 usable t <= 10, overlap R at T
        noise = np.full((n_smear, n_pf), np.nan)
        R_T, R_T_err, T_used, t_last = np.full((n_smear, n_pf), np.nan), np.full((n_smear, n_pf), np.nan), {}, {}
        for p in range(n_pf):
            med = np.nanmedian(e[:, p], axis=0)
            usable = [t for t in range(t_norm, 11) if np.isfinite(med[t]) and med[t] <= usable_err]
            if not usable:
                continue
            win = usable[-4:]
            t_last[p] = usable[-1]
            with np.errstate(invalid="ignore", divide="ignore"):
                noise[:, p] = np.exp(np.nanmean(np.log(e[:, p, win] / med[win][None]), axis=1))
            T = min(7, usable[-1] + 1)
            if T >= t_norm + 2:                                             # R at t_norm + 1 is one step of E_eff: no overlap information
                T_used[p] = T
                R_T[:, p], R_T_err[:, p] = r[:, p, T], re[:, p, T]

        with PdfPages(f"{out_dir}/ratio_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_all, r[:, p], re[:, p], style,
                     f"{particle}  pz = {pf_list[p][2]}   {what}   [C(t)/C({t_norm})] / same for op {i_ref}{note0[p]}",
                     f"ratio -> ground-state fraction at t = {t_norm} relative to op {i_ref}   (> 1: more overlap)",
                     range_of(r[:, p, 1:t_max_plot + 1], re[:, p, 1:t_max_plot + 1], diff_err_max), 1.0, None, diff_err_max)

        #overlap map: all pz on one page
        fig, axes = plt.subplots(2, 4, figsize=(30, 15), constrained_layout=True)
        for p in range(n_pf):
            ax = axes.flat[p]
            ax.grid(alpha=0.3)
            ax.axhline(1, color="gray", lw=1)
            ax.axvline(1, color="gray", lw=1, ls=":")
            ax.axvline(noise_cut, color="gray", lw=1, ls="--")
            ok = np.isfinite(R_T[:, p]) & np.isfinite(noise[:, p])
            x_lo = min(0.5, 0.8 * np.min(noise[ok, p], initial=0.5))
            for i in np.where(ok & (noise[:, p] <= map_noise_max))[0]:
                color, marker, label = style[i]
                x = noise[i, p]
                ax.errorbar(x, R_T[i, p], R_T_err[i, p], fmt=marker, color=color, ms=14 if marker == "*" else 10, mec="black", mew=0.8, elinewidth=1.2, capsize=0)
                ax.annotate(str(i), (x, R_T[i, p]), xytext=(5, 4), textcoords="offset points", fontsize=11)
            hidden = np.where(ok & (noise[:, p] > map_noise_max))[0]
            missing = np.where(~ok)[0] if p in T_used else []
            note = []
            if hidden.size:
                note.append(f"noise > {map_noise_max:g}, not drawn: " + ", ".join(f"{i} ({noise[i, p]:.0f})" for i in hidden))
            if len(missing):
                note.append("no R or noise (GEVP NaN): " + ", ".join(map(str, missing)))
            if note:
                ax.text(0.98, 0.02, "\n".join(note), transform=ax.transAxes, ha="right", va="bottom", fontsize=12,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
            ax.set_xscale("log")
            ticks = [x_t for x_t in [0.2, 0.3, 0.5, 0.7, 1, noise_cut, 1.5, 2, 3] if x_t >= x_lo]
            ax.set_xticks(ticks)
            ax.set_xticklabels([f"{x_t:g}" for x_t in ticks])
            ax.minorticks_off()
            ax.set_xlim(x_lo, map_noise_max)                                # after set_xticks, which widens the view to its ticks
            ax.tick_params(labelsize=14)
            ax.set_title(f"pz = {pf_list[p][2]}   (R at T = {T_used.get(p, '-')})" + (note0[p] and "  no ground-state overlap"), fontsize=17)
            if p % 4 == 0:
                ax.set_ylabel(f"overlap: ground-state fraction at t = {t_norm} relative to op {i_ref}", fontsize=15)
            if p >= 3:
                ax.set_xlabel("noise: E_eff error / median over the operators", fontsize=15)
        leg = axes.flat[7]
        leg.axis("off")
        handles = [plt.Line2D([], [], marker=m, ls="", color=c, mec="black", mew=0.8, ms=14 if m == "*" else 10, label=l) for c, m, l in style]
        leg.legend(handles=handles, loc="center", fontsize=13, frameon=False)
        fig.suptitle(f"{particle}   {what}   {n_cfg} cfgs: top-left is best (more overlap, less noise)", fontsize=22)
        with PdfPages(f"{out_dir}/overlap_map_{name}.pdf") as pdf:
            pdf.savefig(fig)
        plt.close(fig)

        print(f"  {name}: op:R/noise, R = overlap relative to op {i_ref} at T, sorted by R among the operators with noise <= {noise_cut:g}")
        for p in range(n_pf):
            if p not in T_used:
                print(f"    pz {pf_list[p][2]}: " + (f"usable t only up to {t_last[p]}, T = {t_last[p] + 1} < t_norm + 2: no R" if p in t_last else "no usable t"))
                continue
            good = [i for i in np.argsort(-R_T[:, p]) if np.isfinite(R_T[i, p]) and noise[i, p] <= noise_cut]
            bad = [i for i in range(n_smear) if noise[i, p] > noise_cut]
            missing = [i for i in range(n_smear) if i not in good and i not in bad]
            print(f"    pz {pf_list[p][2]} (T = {T_used[p]}){note0[p]}: " + "  ".join(f"{i}:{R_T[i, p]:.3f}/{noise[i, p]:.2f}" for i in good[:7])
                  + f"   | noise > {noise_cut:g}: " + " ".join(map(str, bad)) + ("   | no R or noise (GEVP NaN): " + " ".join(map(str, missing)) if missing else ""))
    print(f"  wrote {out_dir}/")


pion: 177 cfgs, 20 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 19 = [5.         1.66666667 0.8       ]
  GEVP operator 14, pz 6: C(t0) not positive definite (eigenvalues / largest [-0.011956  0.034013  1.      ]), left NaN
  GEVP operator 15, pz 5: C(t0) not positive definite (eigenvalues / largest [-7.6700e-04  7.4407e-02  1.0000e+00]), left NaN
  GEVP operator 15, pz 6: C(t0) not positive definite (eigenvalues / largest [-0.066761  0.023176  1.      ]), left NaN
  G5: op:R/noise, R = overlap relative to op 19 at T, sorted by R among the operators with noise <= 1.2
    pz 0 (T = 7): 13:1.014/1.09  17:1.010/1.15  12:1.004/1.00  16:1.004/1.06  11:1.004/1.11  19:1.000/1.10  10:1.000/1.05   | noise > 1.2: 14 15 18
    pz 1 (T = 7): 18:1.031/1.17  13:1.026/1.19  17:1.020/1.09  12:1.008/1.09  11:1.008/1.07  16:1.008/1.02  10:1.000/1.02   | noise > 1.2: 14 15
    pz 2 (T = 7): 18:1.057/0.98  17:1.039/1.00  11:1.018/0.71  16:1.018/1.00  10:1.006/0.7